# 11주차 실습 — 자연어처리 딥러닝 Ⅰ
## 딥러닝 기초 · 단어벡터를 이용한 추론(word2vec) · 합성곱 신경망(CNN)

In [ ]:
# [0] 환경 설정  ─ 런타임을 새로 시작했다면 이 셀을 가장 먼저 실행한다.
#     GPU 사용 권장 : 런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_FONT = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
try:
    fm.fontManager.addfont(_FONT)                 # 캐시를 거치지 않고 직접 등록
    plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False
    print('한글 폰트 설정 완료 :', plt.rcParams['font.family'])
except Exception as e:
    print('[경고] 한글 폰트 설정 실패 — 그래프의 한글이 네모(▯)로 보입니다 :', e)
    print('       셀을 한 번 더 실행하거나, 그래프 라벨을 영문으로 바꾸어 진행하세요.')


### [1] 실습 진행 관련 설명

11주차 강의 내용에 대한 실습을 진행한다. 실습은 크게 두 가지로 구성되어 있다.

- **실습 1 : word2vec(CBOW)을 NumPy로 직접 구현하기**
  - 계층(MatMul, Softmax with Loss)을 직접 만들고, 순전파와 역전파를 손으로 구현한다.
  - 학습이 끝난 가중치 `W_in`의 각 행이 단어 벡터가 된다는 것을 눈으로 확인한다.
- **실습 2 : 1차원 합성곱 신경망(Text-CNN)으로 감성 분석하기**
  - TensorFlow/Keras의 `Embedding → Conv1D → GlobalMaxPooling1D → Dense` 구조를 사용한다.

먼저 라이브러리를 이용하지 않고 직접 구현하여 원리를 이해한 뒤, 라이브러리를 이용한 간단한 구현을 실습한다.

### [2] 코드 구현의 흐름

**실습 1 (word2vec)**

- (1) (데이터 측면) 말뭉치 불러오기 → 단어 사전(word_to_id, id_to_word) 만들기
- (2) (데이터 측면) 윈도우를 미끄러뜨려 맥락(contexts)과 타깃(target) 만들기 → 원핫 변환
- (3) (모델 측면) MatMul 계층, Softmax with Loss 계층을 직접 구현
- (4) (모델 측면) 두 계층을 조합해 CBOW 모델 정의 (forward / backward)
- (5) (학습 과정) 미니배치 경사하강법(Adam)으로 손실을 줄여 나가기
- (6) (성능 평가) 손실 곡선, 코사인 유사도, 유사 단어 검색, 2차원 시각화

**실습 2 (Text-CNN)**

- (1) (데이터 측면) 리뷰 데이터 불러오기 → 정수 인코딩 → 패딩으로 길이 맞추기
- (2) (모델 측면) Embedding → Conv1D → GlobalMaxPooling1D → Dense 모델 구성
- (3) (학습 과정) 조기 종료(EarlyStopping)를 적용해 학습, 학습 곡선 관찰
- (4) (성능 평가) 평가 데이터 정확도, 혼동 행렬, 새로운 문장에 대한 예측

---
## (공통) 환경 준비

- 구글 코랩에서 실행하는 것을 권장한다. (런타임 → 모두 실행)
- 아래 셀은 그래프에 한글이 깨지지 않도록 한글 폰트를 설치한다. **코랩에서 처음 한 번만 실행**하면 된다.
- 폰트 설치 후 그래프의 한글이 여전히 깨지면 `런타임 → 세션 다시 시작` 후 다시 실행한다.

In [ ]:
# (코랩 전용) 한글 폰트 설치 — 로컬 환경에서는 실행하지 않아도 된다.
!apt-get -qq -y install fonts-nanum > /dev/null

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (설치되어 있는 폰트를 자동으로 찾는다)
def set_korean_font():
    for cand in ['NanumBarunGothic', 'NanumGothic', 'Malgun Gothic',
                 'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']:
        if any(cand == f.name for f in fm.fontManager.ttflist):
            plt.rcParams['font.family'] = cand
            print('한글 폰트 설정 완료 :', cand)
            return True
    print('한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.')
    return False

# 방금 설치한 폰트를 인식시키기 위해 폰트 목록을 다시 읽는다.
for fpath in fm.findSystemFonts(fontpaths=None, fontext='ttf'):
    if 'Nanum' in fpath:
        fm.fontManager.addfont(fpath)

set_korean_font()
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

np.random.seed(42)   # 결과 재현을 위한 난수 고정
print('numpy version :', np.__version__)

---
# 실습 1. word2vec (CBOW) 직접 구현하기

## (1) (데이터 측면) 말뭉치 준비

분포 가설을 확인하기 위해, **같은 자리에 등장하는 단어끼리 의미가 비슷해지는** 작은 한국어 말뭉치를 만든다.

- `과일` 그룹의 단어들은 "먹었다 / 맛있다" 같은 맥락에 등장한다.
- `동물` 그룹의 단어들은 "귀엽다 / 보았다" 같은 맥락에 등장한다.
- `장소` 그룹의 단어들은 "갔다 / 있다" 같은 맥락에 등장한다.

학습이 끝난 뒤 같은 그룹의 단어들이 서로 가까운 벡터를 갖게 되는지 확인하는 것이 목표이다.

In [ ]:
# (1) (데이터 측면) 말뭉치 만들기

과일 = ['사과', '딸기', '바나나', '포도']
동물 = ['강아지', '고양이', '토끼', '병아리']
장소 = ['학교', '도서관', '공원', '시장']

문장틀 = []
for w in 과일:
    문장틀 += ['나는 %s 를 먹었다 .' % w, '그 %s 는 정말 맛있다 .' % w, '엄마가 %s 를 사왔다 .' % w]
for w in 동물:
    문장틀 += ['나는 %s 를 보았다 .' % w, '그 %s 는 정말 귀엽다 .' % w, '동생이 %s 를 키운다 .' % w]
for w in 장소:
    문장틀 += ['나는 %s 에 갔다 .' % w, '그 %s 는 정말 넓다 .' % w, '동생이 %s 에 있다 .' % w]

# 말뭉치를 여러 번 반복해 학습에 충분한 데이터 수를 확보한다.
corpus_text = ' '.join(문장틀 * 3)

print('문장 수 :', len(문장틀))
print('말뭉치의 앞부분 :')
print(corpus_text[:120], '...')

## (1-2) (데이터 측면) 단어 사전 만들기

토큰(단어) → 고유 번호(ID)로 바꾸는 사전을 만든다.

- `word_to_id` : 단어 → ID
- `id_to_word` : ID → 단어
- `corpus` : 말뭉치를 ID의 배열로 바꾼 것

In [ ]:
# (1-2) (데이터 측면) 전처리 : 토큰화 + 단어 사전 구축

def preprocess(text):
    words = text.split(' ')          # 이미 공백으로 분리된 형태이므로 split 사용
    words = [w for w in words if w]  # 빈 문자열 제거

    word_to_id, id_to_word = {}, {}
    for word in words:
        if word not in word_to_id:
            new_id = len(word_to_id)
            word_to_id[word] = new_id
            id_to_word[new_id] = word

    corpus = np.array([word_to_id[w] for w in words])
    return corpus, word_to_id, id_to_word

corpus, word_to_id, id_to_word = preprocess(corpus_text)
vocab_size = len(word_to_id)

print('전체 토큰 수 (말뭉치 길이) :', len(corpus))
print('어휘 수 (vocabulary size) :', vocab_size)
print('단어 사전의 앞부분 :', dict(list(word_to_id.items())[:10]))
print('corpus 의 앞부분 :', corpus[:15])

## (2) (데이터 측면) 맥락(contexts)과 타깃(target) 만들기

윈도우 크기가 1이면, 각 단어를 타깃으로 삼고 그 **좌우 1개**의 단어를 맥락으로 사용한다.

```
... 나는  사과  를  먹었다 ...
      ↑    ↑    ↑
    맥락  타깃  맥락
```

In [ ]:
# (2) (데이터 측면) 맥락과 타깃 만들기

def create_contexts_target(corpus, window_size=1):
    # 양쪽 끝의 window_size 개는 맥락을 모두 확보할 수 없으므로 타깃에서 제외한다.
    target = corpus[window_size:-window_size]
    contexts = []

    for idx in range(window_size, len(corpus) - window_size):
        cs = []
        for t in range(-window_size, window_size + 1):
            if t == 0:            # 자기 자신(타깃)은 맥락에 넣지 않는다.
                continue
            cs.append(corpus[idx + t])
        contexts.append(cs)

    return np.array(contexts), np.array(target)

window_size = 1
contexts, target = create_contexts_target(corpus, window_size)

print('contexts.shape :', contexts.shape, '  (샘플 수, 2*윈도우)')
print('target.shape   :', target.shape)
print()
for i in range(3):
    ctx = [id_to_word[c] for c in contexts[i]]
    print('맥락 %s -> 타깃 %s' % (ctx, id_to_word[target[i]]))

## (2-2) (데이터 측면) 원핫 표현으로 변환

신경망의 입력으로 넣기 위해 ID를 원핫 벡터로 바꾼다.

> 실제 word2vec 구현에서는 이 단계를 **Embedding 계층(행 조회)** 으로 대체하여 메모리와 연산을 아낀다.
> 여기서는 원리를 명확히 보기 위해 원핫 표현을 사용한다.

In [ ]:
# (2-2) (데이터 측면) 원핫 변환

def convert_one_hot(corpus, vocab_size):
    N = corpus.shape[0]

    if corpus.ndim == 1:                 # 타깃 : (N,) -> (N, V)
        one_hot = np.zeros((N, vocab_size), dtype=np.int32)
        for idx, word_id in enumerate(corpus):
            one_hot[idx, word_id] = 1

    elif corpus.ndim == 2:               # 맥락 : (N, C) -> (N, C, V)
        C = corpus.shape[1]
        one_hot = np.zeros((N, C, vocab_size), dtype=np.int32)
        for idx_0, word_ids in enumerate(corpus):
            for idx_1, word_id in enumerate(word_ids):
                one_hot[idx_0, idx_1, word_id] = 1

    return one_hot

target_oh = convert_one_hot(target, vocab_size)
contexts_oh = convert_one_hot(contexts, vocab_size)

print('contexts(one-hot).shape :', contexts_oh.shape)
print('target(one-hot).shape   :', target_oh.shape)
print()
print('첫 번째 타깃 "%s" 의 원핫 벡터 :' % id_to_word[target[0]])
print(target_oh[0])

## (3) (모델 측면) 계층 직접 구현하기

신경망을 **계층(layer) 단위**로 쪼개어 구현한다. 각 계층은 다음 두 메서드를 갖는다.

- `forward(x)` : 순전파. 입력을 받아 출력을 계산한다.
- `backward(dout)` : 역전파. 출력쪽에서 온 기울기를 받아 입력쪽 기울기를 계산하고, 자신의 파라미터 기울기를 저장한다.

In [ ]:
# (3-1) (모델 측면) MatMul 계층 : 완전연결(행렬곱) 연산

class MatMul:
    def __init__(self, W):
        self.params = [W]                 # 학습할 파라미터
        self.grads = [np.zeros_like(W)]   # 파라미터의 기울기
        self.x = None

    def forward(self, x):
        W, = self.params
        self.x = x
        return np.dot(x, W)               # out = x·W

    def backward(self, dout):
        W, = self.params
        dx = np.dot(dout, W.T)            # dL/dx = dout·W^T
        dW = np.dot(self.x.T, dout)       # dL/dW = x^T·dout
        self.grads[0][...] = dW           # 기울기를 덮어써서 저장
        return dx

In [ ]:
# (3-2) (모델 측면) Softmax 와 교차 엔트로피 손실

def softmax(x):
    # 오버플로 방지를 위해 최댓값을 빼준다.
    if x.ndim == 2:
        x = x - x.max(axis=1, keepdims=True)
        x = np.exp(x)
        x /= x.sum(axis=1, keepdims=True)
    else:
        x = x - np.max(x)
        x = np.exp(x) / np.sum(np.exp(x))
    return x

def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    # 정답이 원핫 벡터라면 정답 레이블의 인덱스로 변환한다.
    if t.size == y.size:
        t = t.argmax(axis=1)

    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

class SoftmaxWithLoss:
    def __init__(self):
        self.params, self.grads = [], []
        self.y = None   # softmax 의 출력
        self.t = None   # 정답 레이블

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        if self.t.size == self.y.size:      # 원핫이면 인덱스로 변환
            self.t = self.t.argmax(axis=1)
        return cross_entropy_error(self.y, self.t)

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = self.y.copy()
        dx[np.arange(batch_size), self.t] -= 1    # (y - t)
        dx *= dout
        dx = dx / batch_size
        return dx

## (4) (모델 측면) CBOW 모델 정의

- 입력측 가중치 `W_in` **(V×D)** 은 모든 맥락 단어가 **공유**한다.
- 은닉층은 각 맥락의 결과를 **평균**한 값이다.
- 출력측 가중치 `W_out` **(D×V)** 을 지나 소프트맥스로 어휘 전체의 확률을 만든다.

In [ ]:
# (4) (모델 측면) SimpleCBOW 모델

class SimpleCBOW:
    def __init__(self, vocab_size, hidden_size):
        V, D = vocab_size, hidden_size

        # 가중치 초기화 (작은 무작위 값)
        W_in  = 0.01 * np.random.randn(V, D).astype('f')
        W_out = 0.01 * np.random.randn(D, V).astype('f')

        # 계층 생성 : 맥락이 2개이므로 입력측 MatMul 계층도 2개 (가중치는 공유)
        self.in_layer0 = MatMul(W_in)
        self.in_layer1 = MatMul(W_in)
        self.out_layer = MatMul(W_out)
        self.loss_layer = SoftmaxWithLoss()

        # 모든 파라미터와 기울기를 하나의 리스트에 모은다.
        layers = [self.in_layer0, self.in_layer1, self.out_layer]
        self.params, self.grads = [], []
        for layer in layers:
            self.params += layer.params
            self.grads  += layer.grads

        # 단어의 분산 표현 : 학습이 끝나면 이 행렬의 각 행이 단어 벡터가 된다.
        self.word_vecs = W_in

    def forward(self, contexts, target):
        # 이 모델은 맥락이 2개(window_size=1)인 경우만 처리한다.
        assert contexts.shape[1] == 2, 'SimpleCBOW 는 window_size=1 에서만 동작합니다. (과제의 GeneralCBOW 사용)'
        h0 = self.in_layer0.forward(contexts[:, 0])   # 왼쪽 맥락
        h1 = self.in_layer1.forward(contexts[:, 1])   # 오른쪽 맥락
        h = (h0 + h1) * 0.5                           # 은닉층 = 평균
        score = self.out_layer.forward(h)
        loss = self.loss_layer.forward(score, target)
        return loss

    def backward(self, dout=1):
        ds = self.loss_layer.backward(dout)
        da = self.out_layer.backward(ds)
        da *= 0.5                                     # 평균 연산의 역전파
        self.in_layer1.backward(da)
        self.in_layer0.backward(da)
        return None

## (5) (학습 과정) 옵티마이저와 학습 루프

- 옵티마이저는 강의에서 다룬 **Adam** 을 직접 구현한다.
- 매 에폭마다 데이터를 뒤섞고(shuffle), 미니배치 단위로 기울기를 계산해 파라미터를 갱신한다.

> **주의 : 공유 가중치의 기울기 처리**
> CBOW의 두 입력측 계층은 `W_in` 을 **공유**하므로, `params` 리스트에 같은 배열이 두 번 들어간다.
> 이때 기울기를 각각 갱신하면 `W_in` 이 한 번의 반복에서 두 번 갱신되어 실효 학습률이 2배가 된다.
> 따라서 **중복된 파라미터의 기울기를 먼저 더한 뒤 한 번만 갱신**해야 한다.

In [ ]:
# (5-1) (학습 과정) Adam 옵티마이저 직접 구현

def remove_duplicate(params, grads):
    """공유된(중복된) 파라미터를 하나로 모으고, 그 기울기는 더해 준다."""
    params, grads = params[:], grads[:]
    while True:
        find_flg = False
        for i in range(len(params) - 1):
            for j in range(i + 1, len(params)):
                if params[i] is params[j]:        # 같은 배열을 공유하는 경우
                    grads[i] += grads[j]          # 기울기를 더한다
                    find_flg = True
                    params.pop(j); grads.pop(j)
                    break
            if find_flg: break
        if not find_flg: break
    return params, grads


class Adam:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999):
        self.lr, self.beta1, self.beta2 = lr, beta1, beta2
        self.iter = 0
        self.m, self.v = None, None

    def update(self, params, grads):
        if self.m is None:
            self.m = [np.zeros_like(p) for p in params]
            self.v = [np.zeros_like(p) for p in params]

        self.iter += 1
        lr_t = self.lr * np.sqrt(1.0 - self.beta2 ** self.iter) / (1.0 - self.beta1 ** self.iter)

        for i in range(len(params)):
            self.m[i] += (1 - self.beta1) * (grads[i] - self.m[i])
            self.v[i] += (1 - self.beta2) * (grads[i] ** 2 - self.v[i])
            params[i] -= lr_t * self.m[i] / (np.sqrt(self.v[i]) + 1e-7)

In [ ]:
# (5-2) (학습 과정) 학습 루프

def train_cbow(model, optimizer, contexts, target, max_epoch=300, batch_size=64, verbose_every=50):
    data_size = len(contexts)
    max_iters = max(1, data_size // batch_size)   # 마지막 자투리 배치는 사용하지 않는다
    loss_history = []

    for epoch in range(max_epoch):
        # 1) 데이터를 뒤섞는다 (미니배치가 매번 달라지도록)
        idx = np.random.permutation(data_size)
        c, t = contexts[idx], target[idx]

        total_loss = 0
        for iters in range(max_iters):
            batch_c = c[iters * batch_size:(iters + 1) * batch_size]
            batch_t = t[iters * batch_size:(iters + 1) * batch_size]

            loss = model.forward(batch_c, batch_t)   # 2) 순전파 : 손실 계산
            model.backward()                         # 3) 역전파 : 기울기 계산

            # 4) 공유 가중치의 기울기를 합산한 뒤 파라미터 갱신
            params, grads = remove_duplicate(model.params, model.grads)
            optimizer.update(params, grads)
            total_loss += loss

        avg_loss = total_loss / max_iters
        loss_history.append(avg_loss)
        if (epoch + 1) % verbose_every == 0:
            print('epoch %3d / %3d  |  loss %.4f' % (epoch + 1, max_epoch, avg_loss))

    return loss_history

hidden_size = 10      # 단어 벡터의 차원 D
max_epoch   = 300
batch_size  = 64

model = SimpleCBOW(vocab_size, hidden_size)
optimizer = Adam(lr=0.01)

loss_history = train_cbow(model, optimizer, contexts_oh, target_oh,
                          max_epoch=max_epoch, batch_size=batch_size)

## (6) (성능 평가) 학습 결과 확인

In [ ]:
# (6-1) (성능 평가) 손실 곡선

plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='tab:blue')
plt.title('CBOW 학습 손실')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.grid(alpha=0.3)
plt.show()

print('첫 에폭 손실 : %.4f' % loss_history[0])
print('마지막 에폭 손실 : %.4f' % loss_history[-1])

In [ ]:
# (6-2) (성능 평가) 학습된 단어 벡터 확인
#  W_in 의 i 번째 행 = i 번째 단어의 분산 표현

word_vecs = model.word_vecs
print('단어 벡터 행렬의 모양 :', word_vecs.shape, ' (어휘 수 V, 차원 D)')
print()
for word in ['사과', '강아지', '학교']:
    wid = word_to_id[word]
    print('%-5s ->' % word, np.round(word_vecs[wid], 3))

In [ ]:
# (6-3) (성능 평가) 코사인 유사도와 유사 단어 검색

def cos_similarity(x, y, eps=1e-8):
    nx = x / (np.sqrt(np.sum(x ** 2)) + eps)   # x 의 정규화
    ny = y / (np.sqrt(np.sum(y ** 2)) + eps)   # y 의 정규화
    return np.dot(nx, ny)

def most_similar(query, word_to_id, id_to_word, word_matrix, top=5, verbose=True):
    if query not in word_to_id:
        if verbose:
            print(f'{query} 를 찾을 수 없습니다.')
        return []

    if verbose:
        print('[query] ' + query)

    query_vec = word_matrix[word_to_id[query]]

    vocab_size = len(id_to_word)
    similarity = np.zeros(vocab_size)

    for i in range(vocab_size):
        similarity[i] = cos_similarity(word_matrix[i], query_vec)

    results = []

    for i in (-similarity).argsort():
        if id_to_word[i] == query:
            continue

        results.append((id_to_word[i], float(similarity[i])))

        if verbose:
            print('  %-6s : %.4f' % (id_to_word[i], similarity[i]))

        if len(results) >= top:
            break

    return results

for q in ['사과', '고양이', '도서관']:
    most_similar(q, word_to_id, id_to_word, word_vecs, top=5)
    print()

> **관찰 포인트**
> - `사과`의 유사 단어로 다른 과일이, `고양이`의 유사 단어로 다른 동물이 올라오는지 확인해 보자.
> - 이것이 바로 **분포 가설**(문맥이 비슷한 단어는 의미도 비슷하다)이 학습에 반영된 결과이다.
> - 말뭉치가 작으므로 완벽하지는 않다. 말뭉치를 늘리거나 에폭을 늘리면 개선된다.

In [ ]:
# (6-4) (성능 평가) 단어 벡터를 2차원으로 투영해 시각화
#  PCA(주성분 분석)를 특이값 분해(SVD)로 직접 구현한다.

def pca_2d(X):
    Xc = X - X.mean(axis=0)              # 평균을 0으로 맞춤
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T                 # 상위 2개 주성분에 투영

xy = pca_2d(word_vecs)

# 그룹별로 색을 다르게 표시한다.
group_of = {}
for w in 과일:  group_of[w] = ('tab:blue', '과일')
for w in 동물:  group_of[w] = ('tab:orange', '동물')
for w in 장소:  group_of[w] = ('tab:green', '장소')

plt.figure(figsize=(9, 7))
shown = set()
for word, wid in word_to_id.items():
    color, label = group_of.get(word, ('lightgray', '기타'))
    plt.scatter(xy[wid, 0], xy[wid, 1], color=color, s=60,
                label=label if label not in shown else None)
    shown.add(label)
    plt.annotate(word, (xy[wid, 0], xy[wid, 1]), fontsize=11,
                 xytext=(4, 4), textcoords='offset points')

plt.title('학습된 단어 벡터의 2차원 투영 (PCA)')
plt.xlabel('주성분 1'); plt.ylabel('주성분 2')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## (7) (심화) 네거티브 샘플링을 위한 확률 분포

강의에서 다룬 것처럼, 어휘가 커지면 소프트맥스 계산이 병목이 된다.
네거티브 샘플링은 **정답 1개 + 무작위 오답 k개**만으로 이진 분류 손실을 계산해 이 문제를 해결한다.

이때 오답은 단어의 출현 빈도를 **0.75 제곱**한 확률 분포에서 뽑는다.
아래에서 0.75 제곱이 저빈도 단어의 확률을 어떻게 끌어올리는지 직접 확인해 보자.

In [ ]:
# (7) (심화) 0.75 제곱의 효과 확인

counts = np.bincount(corpus, minlength=vocab_size).astype(np.float64)

p_raw  = counts / counts.sum()                      # 빈도를 그대로 사용
p_075  = (counts ** 0.75) / np.sum(counts ** 0.75)  # 0.75 제곱 적용

order = np.argsort(-p_raw)[:12]
labels = [id_to_word[i] for i in order]

x = np.arange(len(order))
plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, p_raw[order], width=0.4, label='빈도 그대로')
plt.bar(x + 0.2, p_075[order], width=0.4, label='0.75 제곱 적용')
plt.xticks(x, labels, rotation=45)
plt.ylabel('샘플링 확률')
plt.title('네거티브 샘플링 확률 분포의 비교')
plt.legend(); plt.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print('가장 흔한 단어의 확률 : %.4f -> %.4f (감소)' % (p_raw[order[0]], p_075[order[0]]))
print('가장 드문 단어의 확률 : %.4f -> %.4f (증가)' % (p_raw.min(), p_075[p_raw.argmin()]))

---
# 실습 2. 1차원 합성곱 신경망(Text-CNN)으로 감성 분석하기

`Embedding → Conv1D → GlobalMaxPooling1D → Dense` 구조를 만들어 영화 리뷰의 긍정/부정을 분류한다.

- 데이터 : 케라스에 내장된 **IMDB 영화 리뷰** (긍정/부정 각 25,000건, 이미 정수로 인코딩되어 있음)
- 별도의 파일 다운로드가 필요 없으므로 코랩에서 바로 실행할 수 있다.

> ⚠️ **실행 전 확인** : 코랩 상단 메뉴에서 `런타임 → 런타임 유형 변경 → 하드웨어 가속기 : T4 GPU` 로 설정한다.
> CPU 런타임에서는 학습에 수십 분이 걸릴 수 있다. (GPU 사용 시 에폭당 10초 내외)

In [ ]:
# (1) (데이터 측면) 라이브러리와 데이터 불러오기

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
# pad_sequences 의 위치는 케라스 버전에 따라 다르므로 두 경우를 모두 처리한다.
try:
    from tensorflow.keras.utils import pad_sequences        # TF 2.9 / Keras 2.9 이상
except ImportError:
    from tensorflow.keras.preprocessing.sequence import pad_sequences

keras.utils.set_random_seed(42)      # 파이썬·넘파이·텐서플로 시드를 한 번에 고정
print('tensorflow version :', tf.__version__)

VOCAB_SIZE = 10000   # 빈도 상위 10,000 단어만 사용 (나머지는 <UNK> 처리)
MAX_LEN    = 200     # 리뷰 하나의 최대 길이 (단어 수)

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print('학습 데이터 수 :', len(x_train), ' / 평가 데이터 수 :', len(x_test))
print('첫 번째 리뷰의 앞부분(정수 인코딩) :', x_train[0][:15])
print('첫 번째 리뷰의 정답(1=긍정, 0=부정) :', y_train[0])

In [ ]:
# (1-2) (데이터 측면) 정수 인덱스를 실제 단어로 되돌려 확인해 보기

word_index = keras.datasets.imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word[0], index_to_word[1], index_to_word[2] = '<PAD>', '<START>', '<UNK>'

def decode_review(seq):
    return ' '.join(index_to_word.get(i, '?') for i in seq)

print('[원문 복원]')
print(decode_review(x_train[0])[:400], '...')
print()
print('리뷰 길이의 분포 : 최소 %d, 최대 %d, 평균 %.1f'
      % (min(map(len, x_train)), max(map(len, x_train)),
         np.mean([len(s) for s in x_train])))

## (2) (데이터 측면) 패딩으로 길이 맞추기

신경망은 고정된 크기의 입력을 받으므로, 모든 리뷰의 길이를 `MAX_LEN`으로 맞춘다.

- 짧은 리뷰 : 앞쪽에 0(`<PAD>`)을 채운다.
- 긴 리뷰 : 앞쪽을 잘라낸다.

In [ ]:
# (2) (데이터 측면) 패딩

x_train_pad = pad_sequences(x_train, maxlen=MAX_LEN, padding='pre', truncating='pre')
x_test_pad  = pad_sequences(x_test,  maxlen=MAX_LEN, padding='pre', truncating='pre')

y_train = np.array(y_train)
y_test  = np.array(y_test)

print('x_train_pad.shape :', x_train_pad.shape, '  (샘플 수, 문장 길이)')
print('x_test_pad.shape  :', x_test_pad.shape)
print()
print('패딩된 첫 번째 리뷰의 앞부분 :', x_train_pad[0][:20])

## (3) (모델 측면) Text-CNN 모델 구성

| 층 | 역할 |
|---|---|
| `Embedding` | 단어 ID를 `EMBED_DIM` 차원의 밀집 벡터로 변환 (word2vec 과 같은 역할) |
| `Conv1D` | 단어 축으로 이동하는 1차원 커널 → n-gram 패턴 탐지 |
| `GlobalMaxPooling1D` | 각 특징 맵의 최대값만 남겨 문장 길이와 무관한 고정 길이 벡터 생성 |
| `Dropout(0.5)` | 과적합 억제 |
| `Dense(64, relu)` | 추출된 특징을 한 번 더 비선형 변환 |
| `Dropout(0.5)` | 과적합 억제 |
| `Dense(1, sigmoid)` | 긍정/부정 이진 분류 |

In [ ]:
# (3) (모델 측면) 모델 구성

EMBED_DIM  = 100    # 단어 임베딩 차원 D
NUM_FILTER = 128    # 합성곱 필터 수
KERNEL_SZ  = 3      # 커널 크기 (= 3-gram 패턴을 본다)

def build_textcnn(kernel_size=KERNEL_SZ, num_filters=NUM_FILTER, embed_dim=EMBED_DIM):
    model = keras.Sequential([
        layers.Input(shape=(MAX_LEN,), dtype='int32'),          # 입력 : 정수 시퀀스
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=embed_dim),
        layers.Conv1D(filters=num_filters, kernel_size=kernel_size,
                      padding='valid', activation='relu'),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),   # 이진 분류이므로 출력 1개 + sigmoid
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

model_cnn = build_textcnn()
model_cnn.summary()

> **출력 크기 직접 확인해 보기**
> - Embedding 출력 : (None, 200, 100) — (문장 길이, 임베딩 차원)
> - Conv1D 출력 : (None, 198, 128) — 200 − 3 + 1 = **198**, 필터 수 128
> - GlobalMaxPooling1D 출력 : (None, 128) — 특징 맵마다 최대값 1개
>
> 강의에서 배운 출력 크기 공식 `(입력 − 커널 + 2×패딩) / 스트라이드 + 1` 과 일치하는지 확인하자.

In [ ]:
# (4) (학습 과정) 조기 종료를 적용해 학습

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                           restore_best_weights=True)

history = model_cnn.fit(x_train_pad, y_train,
                        epochs=6,       # 조기 종료가 걸리므로 6 에폭이면 충분하다
                        batch_size=64,
                        validation_split=0.2,   # 학습 데이터의 20%를 검증에 사용
                        callbacks=[early_stop],
                        verbose=1)

In [ ]:
# (5) (성능 평가) 학습 곡선

def plot_history(history):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(h['loss'], label='학습 손실')
    axes[0].plot(h['val_loss'], label='검증 손실', linestyle='--')
    axes[0].set_title('손실'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(h['accuracy'], label='학습 정확도')
    axes[1].plot(h['val_accuracy'], label='검증 정확도', linestyle='--')
    axes[1].set_title('정확도'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()

plot_history(history)

> **관찰 포인트**
> - 학습 손실은 계속 줄어드는데 검증 손실이 어느 시점부터 올라간다면 **과적합**이 시작된 것이다.
> - `EarlyStopping(restore_best_weights=True)` 덕분에 검증 손실이 가장 낮았던 가중치가 복원된다.

In [ ]:
# (6) (성능 평가) 평가 데이터 정확도와 혼동 행렬

loss, acc = model_cnn.evaluate(x_test_pad, y_test, verbose=0)
print('평가 데이터 손실 : %.4f' % loss)
print('평가 데이터 정확도 : %.4f' % acc)

# 혼동 행렬 (직접 계산)
prob = model_cnn.predict(x_test_pad, verbose=0).ravel()
pred = (prob >= 0.5).astype(int)

TP = int(np.sum((pred == 1) & (y_test == 1)))
TN = int(np.sum((pred == 0) & (y_test == 0)))
FP = int(np.sum((pred == 1) & (y_test == 0)))
FN = int(np.sum((pred == 0) & (y_test == 1)))

print()
print('             예측:부정  예측:긍정')
print('실제:부정      %6d    %6d' % (TN, FP))
print('실제:긍정      %6d    %6d' % (FN, TP))
print()
precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)
print('정밀도(precision) : %.4f' % precision)
print('재현율(recall)    : %.4f' % recall)
print('F1 점수           : %.4f' % f1)

In [ ]:
# (7) (성능 평가) 새로운 문장으로 예측해 보기

def encode_text(text):
    # IMDB 규칙에 맞추어 단어를 정수로 변환한다 (인덱스 +3, 미등록 단어는 2=<UNK>)
    ids = [1]   # <START>
    for w in text.lower().split():
        w = ''.join(ch for ch in w if ch.isalnum())
        if not w:                       # 구두점만으로 된 토큰은 건너뛴다
            continue
        idx = word_index.get(w)
        ids.append(idx + 3 if (idx is not None and idx + 3 < VOCAB_SIZE) else 2)
    return ids

samples = [
    "this movie was absolutely wonderful the acting was great and i loved every minute",
    "what a boring and terrible film i wasted two hours of my life",
    "the story is predictable but the visuals are beautiful",
]

seqs = pad_sequences([encode_text(s) for s in samples], maxlen=MAX_LEN)
probs = model_cnn.predict(seqs, verbose=0).ravel()

for s, p in zip(samples, probs):
    label = '긍정' if p >= 0.5 else '부정'
    print('[%s / %.3f] %s' % (label, p, s[:60] + '...'))

## (8) (심화) 여러 크기의 커널을 병렬로 사용하기 (Kim, 2014)

Text-CNN 원 논문(Kim, 2014)은 **3·4·5-gram 커널(각 100개)을 동시에** 사용한 뒤 결과를 이어 붙인다.
Sequential 모델로는 이런 분기 구조를 표현할 수 없으므로 **Functional API** 를 사용한다.

In [ ]:
# (8) (심화) 다중 커널 Text-CNN (Functional API)

def build_multi_kernel_textcnn(kernel_sizes=(3, 4, 5), num_filters=100, embed_dim=EMBED_DIM):
    inputs = keras.Input(shape=(MAX_LEN,), dtype='int32')
    emb = layers.Embedding(VOCAB_SIZE, embed_dim)(inputs)

    pooled = []
    for k in kernel_sizes:                      # 커널 크기별로 분기
        c = layers.Conv1D(num_filters, k, padding='valid', activation='relu')(emb)
        p = layers.GlobalMaxPooling1D()(c)
        pooled.append(p)

    x = layers.Concatenate()(pooled)            # 분기 결과를 이어 붙인다
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_multi = build_multi_kernel_textcnn()
model_multi.summary()

history_multi = model_multi.fit(x_train_pad, y_train, epochs=6, batch_size=64,
                                validation_split=0.2,
                                callbacks=[keras.callbacks.EarlyStopping(
                                    monitor='val_loss', patience=2, restore_best_weights=True)],
                                verbose=1)

loss_m, acc_m = model_multi.evaluate(x_test_pad, y_test, verbose=0)
print()
print('단일 커널(k=3) 정확도 : %.4f' % acc)
print('다중 커널(3,4,5) 정확도   : %.4f' % acc_m)

## (9) (참고) 사전 학습된 단어 임베딩을 넣는 방법

실습 1에서 만든 것과 같은 word2vec 임베딩을 이미 갖고 있다면, `Embedding` 층의 초기값으로 넣어 줄 수 있다.

```python
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
for word, i in word_to_id.items():
    if i < VOCAB_SIZE and word in pretrained:
        embedding_matrix[i] = pretrained[word]     # 사전 학습된 벡터로 초기화

layers.Embedding(VOCAB_SIZE, EMBED_DIM,
                 embeddings_initializer=keras.initializers.Constant(embedding_matrix),
                 trainable=True)    # False 로 두면 임베딩을 고정(freeze)한다
```

- 학습 데이터가 **적을 때** : `trainable=False` 로 고정하는 편이 유리하다.
- 학습 데이터가 **충분할 때** : `trainable=True` 로 두어 미세 조정(fine-tuning)하면 보통 성능이 가장 좋다.

---
# [과제]

아래 두 과제를 수행하고, **코드 · 실행 결과(표 또는 그래프) · 분석**을 함께 제출한다.
`TODO` 로 표시된 부분을 직접 채워 넣으면 된다.

## 과제 1. word2vec 하이퍼파라미터 실험 (실습 1 활용)

윈도우 크기와 임베딩 차원을 바꾸어 학습하고, 결과를 아래 표로 정리한 뒤 분석을 작성한다.

| window_size | hidden_size | 마지막 손실 | `사과`의 유사 단어 top-3 | `고양이`의 유사 단어 top-3 |
|---|---|---|---|---|
| 1 | 10 |  |  |  |
| 2 | 10 |  |  |  |
| 1 | 50 |  |  |  |
| 2 | 50 |  |  |  |

**분석 질문**

1. 윈도우 크기를 키우면 유사 단어의 성격이 어떻게 달라지는가? 그 이유를 분포 가설로 설명하시오.
2. 임베딩 차원을 키우면 손실은 어떻게 변하는가? 손실이 낮아지는 것이 항상 좋은 것인지 서술하시오.
3. 같은 그룹(과일/동물/장소)의 단어가 서로 가깝게 모이지 않은 경우가 있다면 그 원인을 추측하시오.

In [ ]:
# [과제 1] 윈도우 크기와 은닉 차원에 따른 단어 벡터 변화
#  힌트 : create_contexts_target(corpus, window_size) 부터 다시 실행해야 한다.
#        window_size 가 2 이면 맥락이 4개가 되므로, 맥락 수에 관계없이 동작하는
#        아래 GeneralCBOW 클래스를 사용한다.

class GeneralCBOW:
    """맥락 단어 수(2*window_size)에 관계없이 동작하는 CBOW 모델"""
    def __init__(self, vocab_size, hidden_size, context_size):
        V, D = vocab_size, hidden_size
        W_in  = 0.01 * np.random.randn(V, D).astype('f')
        W_out = 0.01 * np.random.randn(D, V).astype('f')

        self.in_layers = [MatMul(W_in) for _ in range(context_size)]  # 가중치 공유
        self.out_layer = MatMul(W_out)
        self.loss_layer = SoftmaxWithLoss()
        self.context_size = context_size

        self.params, self.grads = [], []
        for layer in self.in_layers + [self.out_layer]:
            self.params += layer.params
            self.grads  += layer.grads
        self.word_vecs = W_in

    def forward(self, contexts, target):
        h = 0
        for i, layer in enumerate(self.in_layers):
            h = h + layer.forward(contexts[:, i])
        h = h * (1.0 / self.context_size)
        score = self.out_layer.forward(h)
        return self.loss_layer.forward(score, target)

    def backward(self, dout=1):
        ds = self.loss_layer.backward(dout)
        da = self.out_layer.backward(ds)
        da = da * (1.0 / self.context_size)
        for layer in reversed(self.in_layers):
            layer.backward(da)


def run_cbow_experiment(ws, hs):
    """윈도우 크기 ws, 은닉 차원 hs 로 CBOW 를 학습하고
    (마지막 손실, '사과'와 유사한 단어 3개) 를 돌려준다."""
    np.random.seed(1)
    ctx, tgt = create_contexts_target(corpus, window_size=ws)   # 1) 맥락·타깃
    ctx_oh = convert_one_hot(ctx, vocab_size)                   #    원핫 변환
    tgt_oh = convert_one_hot(tgt, vocab_size)

    m = GeneralCBOW(vocab_size, hs, context_size=2 * ws)        # 2) 모델·최적화
    hist = train_cbow(m, Adam(lr=0.01), ctx_oh, tgt_oh,
                      max_epoch=300, verbose_every=300)

    final_loss = hist[-1]                                       # 3) 결과 기록
    sims = most_similar('사과', word_to_id, id_to_word, m.word_vecs,
                        top=3, verbose=False)
    return final_loss, sims


results = []
for ws in [1, 2]:
    for hs in [10, 50]:
        try:
            loss, sims = run_cbow_experiment(ws, hs)
            results.append({'window': ws, 'hidden': hs,
                            'final_loss': round(float(loss), 4), 'top3': sims})
            print(f'window={ws} hidden={hs} → loss={loss:.4f}  {sims}')
        except NotImplementedError as e:
            print('▶ 미구현 :', e)
            break
    else:
        continue
    break

if results:
    import pandas as pd
    display(pd.DataFrame(results))


### [과제 1] 분석 작성란

(여기에 표와 분석을 작성하시오.)

1.
2.
3.

## 과제 2. Text-CNN 구조 실험 (실습 2 활용)

커널 크기와 필터 수를 바꾸어 학습하고, 결과를 아래 표로 정리한 뒤 분석을 작성한다.

| 모델 | 커널 크기 | 필터 수 | 검증 정확도 | 평가 정확도 | 파라미터 수 |
|---|---|---|---|---|---|
| 단일 커널 | 2 | 128 |  |  |  |
| 단일 커널 | 3 | 128 |  |  |  |
| 단일 커널 | 5 | 128 |  |  |  |
| 다중 커널 | 3,4,5 | 100 × 3 |  |  |  |

**분석 질문**

1. 커널 크기가 커지면 무엇을 더 잘 잡아내는가? 정확도는 어떻게 변했는가?
2. 다중 커널 모델이 단일 커널 모델보다 좋았는가? 파라미터 수 대비 성능 향상은 어떠한가?
3. 학습 곡선을 근거로 각 모델의 과적합 시점을 비교하시오.
4. (선택) `MAX_LEN` 을 100 또는 400으로 바꾸면 결과가 어떻게 달라지는가?

In [ ]:
# [과제 2] 커널 크기에 따른 Text-CNN 성능 비교

kernel_list = [2, 3, 5]
report = []


def train_single_kernel(k):
    """커널 크기 k 인 단일 분기 Text-CNN 을 학습하고
    (검증 정확도, 평가 정확도, 파라미터 수) 를 돌려준다.
    다중 커널 모델과 같은 조건(epochs=6, batch_size=64,
    validation_split=0.2, EarlyStopping)으로 학습해야 비교가 성립한다."""
    m = build_textcnn(kernel_size=k)                       # 1) 모델 생성
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                       restore_best_weights=True)
    h = m.fit(x_train_pad, y_train, epochs=6, batch_size=64,   # 2) 학습
              validation_split=0.2, callbacks=[es], verbose=1)
    val_acc = max(h.history['val_accuracy'])                   # 3) 결과 수집
    test_acc = m.evaluate(x_test_pad, y_test, verbose=0)[1]
    return val_acc, test_acc, m.count_params()


for k in kernel_list:
    print('=' * 50)
    print('kernel_size =', k)
    try:
        val_acc, test_acc, params = train_single_kernel(k)
        report.append({'kernel': k, 'val_acc': round(val_acc, 4),
                       'test_acc': round(test_acc, 4), 'params': params})
    except NotImplementedError as e:
        print('▶ 미구현 :', e)
        break

if report:
    import pandas as pd
    df = pd.DataFrame(report)
    display(df)
    best = df.loc[df['test_acc'].idxmax()]
    print(f"평가 정확도가 가장 높은 커널 : {int(best['kernel'])} ({best['test_acc']:.4f})")


### [과제 2] 분석 작성란

(여기에 표와 분석을 작성하시오.)

1.
2.
3.
4.

---
## 제출 안내

- 제출 파일 : `11주차_실습_학번_이름.ipynb`
- 모든 셀을 **실행한 상태**(출력 포함)로 저장하여 제출한다.
  - 코랩 : `파일 → 다운로드 → .ipynb 다운로드`
- 과제 1, 과제 2의 표와 분석은 반드시 **본인의 실행 결과**를 근거로 작성한다.